# Ejercicio 2


In [ ]:
import simpy
import random
import numpy as np

# Parámetros
np.random.seed(1)
random.seed(1)

LAMBDA = 1 / 5

MU = 1/4

N_CLIENTES = 100000

# Crear entorno simpy
env = simpy.Environment()

chatbots = {
    1: simpy.Resource(env, capacity=2),
    2: simpy.Resource(env, capacity=1),
    3: simpy.Resource(env, capacity=1),
    4: simpy.Resource(env, capacity=1),
    5: simpy.Resource(env, capacity=2)
}

# Métricas
esperas = {i: [] for i in range(1, 6)}
tiempos_nodo = {i: [] for i in range(1, 6)}
ocupacion = {i: 0 for i in range(1, 6)}
visitas_nodo = {i: 0 for i in range(1, 6)}

tiempo_sistema = []

# Tránsito
def siguiente_nodo(actual):

    if actual == 1:
        return np.random.choice([2, 3,5], p=[0.5, 0.4,0.1])

    elif actual == 2:
        return np.random.choice([3, 4], p=[0.6, 0.4])

    elif actual == 3:
        return 4

    elif actual == 4:
        return 5

    elif actual == 5:
        return 6

    return 6

# Cliente
def cliente(env, nombre):

    llegada = env.now
    visitas = 0

    nodo = 1

    while nodo != 6:

        visitas += 1
        visitas_nodo[nodo] += 1

        recurso = chatbots[nodo]

        instante_cola = env.now

        with recurso.request() as req:

            yield req

            espera = env.now - instante_cola
            esperas[nodo].append(espera)

            servicio = random.expovariate(MU)

            yield env.timeout(servicio)

            fin_servicio = env.now

            tiempos_nodo[nodo].append(fin_servicio - instante_cola)
            ocupacion[nodo] += servicio

        nodo = siguiente_nodo(nodo)

    tiempo_sistema.append(env.now - llegada)

# Llegadas
def llegadas(env):

    for i in range(N_CLIENTES):

        env.process(cliente(env, f"C{i}"))

        interarrival = random.expovariate(LAMBDA)
        yield env.timeout(interarrival)

# Ejecutar
env.process(llegadas(env))
env.run()

# Resultados
print("\nTiempo medio de espera")

for i in range(1, 6):
    print(
        f"Chatbot {i}: "
        f"{np.mean(esperas[i]):.3f}"
    )

print("\nUtilización")

tiempo_total = env.now

for i in range(1, 6):

    rho = ocupacion[i] / (tiempo_total * chatbots[i].capacity)

    print(
        f"Chatbot {i}: {rho:.3f}"
    )

print("\nNúmero medio en el sistema")


for i in range(1, 6):
    N = (visitas_nodo[i] / tiempo_total)* np.mean(tiempos_nodo[i])

    print(f"Chatbot {i}: {N:.3f}")

print("\nTiempo medio en sistema")

print(np.mean(tiempo_sistema))



Tiempo medio de espera
Chatbot 1: 0.739
Chatbot 2: 2.725
Chatbot 3: 4.980
Chatbot 4: 10.244
Chatbot 5: 0.751

Utilización
Chatbot 1: 0.400
Chatbot 2: 0.403
Chatbot 3: 0.558
Chatbot 4: 0.721
Chatbot 5: 0.400

Número medio en el sistema
Chatbot 1: 0.947
Chatbot 2: 0.676
Chatbot 3: 1.254
Chatbot 4: 2.567
Chatbot 5: 0.951

Tiempo medio en sistema
31.967012191677384
